**Business Insights using LLM**

In [ ]:
%pip install torch torchvision torchaudio transformers accelerate bitsandbytes --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 11.8 MB/s eta 0:00:00


In [ ]:
import os
import pandas as pd
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
import torch
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

os.environ["HUGGINGFACE_TOKEN"] = ""
from huggingface_hub import login
login(token=os.environ["HUGGINGFACE_TOKEN"])


In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "mistralai/Mistral-7B-Instruct-v0.1"

quant_config = BitsAndBytesConfig(load_in_4bit=True)
tokenizer = AutoTokenizer.from_pretrained(model_name, token=os.environ["HUGGINGFACE_TOKEN"])
model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=quant_config, token=os.environ["HUGGINGFACE_TOKEN"])


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.94G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [5]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

df_dataset = pd.read_csv("feature_engineered_with_SLM.csv")
df_metrics = pd.read_csv("metrics_all_models.csv")

print(f"Dataset shape: {df_dataset.shape}")
print(f"Dataset : {df_dataset.head()}")
print(f"Metrics: {df_metrics}")

Dataset shape: (1200, 33)
Dataset :    transaction_id  customer_id transaction_datetime country       channel  \
0             998         1000  2022-10-05 15:13:00      US    mobile_app   
1             953         1000  2023-02-05 06:18:18      US           web   
2              94         1000  2023-03-23 18:04:02      US  pos_terminal   
3             457         1000  2023-04-05 00:14:27      US    mobile_app   
4            1199         1000  2023-07-13 02:02:46      US    mobile_app   

  merchant_category   amount  device_trust_score  num_txn_24h_customer  \
0            luxury   672.47                23.0                     2   
1            gaming  2593.31                84.0                     0   
2       electronics   302.22                70.0                     3   
3            gaming  4395.07                24.0                     0   
4            luxury   579.99                23.0                     1   

   previous_chargeback_count  ...  txn_hour_count txn_da

In [6]:
schema_dataset = str(df_dataset.dtypes)
metrics_text = df_metrics.to_string(index=False)

In [7]:
prompt = f"""
You are a business analyst. Based on the schema and fraud model performance below,
TASK :
1. List top 3 highest contributing factors for fraud.
2. Provide the business insight for decision maker so less fraud cases will happen.
3. The business insight should be in bullet points.
4. Each bullet points should form a complete sentence
5. Do not stop early


Schema:
{schema_dataset}

Fraud Model Performance:
{metrics_text}

Business Insight:
"""


In [8]:
inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)
outputs = model.generate(**inputs, max_new_tokens=500)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(" LLM Extracted Insight:\n")
print(response[len(prompt):].strip())


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


 LLM Extracted Insight:

1. The top 3 highest contributing factors for fraud are:
   - is_risky_country (int64)
   - low_device_trust (int64)
   - high_amount_z (int64)

2. To reduce fraud cases, the business can consider the following insights:
   - Monitor and flag transactions from high-risk countries.
   - Implement device trust checks to prevent fraud from untrusted devices.
   - Set limits on high-value transactions to reduce the risk of fraud.
   - Implement additional fraud detection measures such as anomaly detection and behavioral analysis.
   - Regularly review and update fraud models to ensure they are up-to-date and effective.
